In [ ]:
method= 'LocalHYSEP'
methods= ['Furey','USGS', 'Eckhardt', 'LocalHYSEP']
models= ['HBV', 'HBV_capillary', 'marrmot_PRMS']
model= 'HBV'
import numpy as np

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den


def calc_kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    std_obs = np.std(obs)
    std_sim = np.std(sim)

    if std_obs == 0:
        return np.nan

    alpha = std_sim / std_obs

    mean_obs = np.mean(obs)
    if mean_obs == 0:
        return np.nan

    beta = np.mean(sim) / mean_obs

    return 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )


def calc_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return (np.sum(sim) - den) / den


def calc_abs_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return np.abs(np.sum(sim) - den) / den


for method in methods:
    for model in models:
        if model=='HBV':
                dyn= 'parBETA_parBETAET_'

        if model=='marrmot_PRMS':
                dyn= 'alpha_cgw_scx_tt_'

        if model=='HBV_capillary':
                dyn= 'parBETA_parBETAET_parK0_parUZL_'


        obs_flow= np.load(f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/00060_Mean.npy")
        single= np.load(f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/OUTPUTS_Temporal_Single/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/flow_sim.npy")
        single=single.squeeze()
        multi= np.load(f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/flow_sim.npy")
        multi= multi.squeeze()


        # ============================================================
        # Containers
        # ============================================================

        metrics_old = {
                "NSE": [],
                "KGE": [],
                "Bias": [],
                "AbsBias": [],
                "NSE_low": [],
                "NSE_high": [],
                "Bias_low": [],
                "Bias_high": [],
                "AbsBias_low": [],
                "AbsBias_high": [],
            }

        metrics_new = {
                "NSE": [],
                "KGE": [],
                "Bias": [],
                "AbsBias": [],
                "NSE_low": [],
                "NSE_high": [],
                "Bias_low": [],
                "Bias_high": [],
                "AbsBias_low": [],
                "AbsBias_high": [],
            }
        # --------------------------------------------------------
        # Loop over basins
        # --------------------------------------------------------
        print(obs_flow.shape)
        for k in range(obs_flow.shape[1]):

                    obs = obs_flow[:, k]
                    old = single[:, k]
                    new = multi[:, k]

                    # =====================================================
                    # Overall metrics
                    # =====================================================
                    metrics_old["NSE"].append(calc_nse(obs, old))
                    metrics_new["NSE"].append(calc_nse(obs, new))

                    metrics_old["KGE"].append(calc_kge(obs, old))
                    metrics_new["KGE"].append(calc_kge(obs, new))

                    metrics_old["Bias"].append(calc_bias(obs, old))
                    metrics_new["Bias"].append(calc_bias(obs, new))

                    metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
                    metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

                    # =====================================================
                    # Low / High flow
                    # =====================================================
                    threshold = np.median(obs)

                    low = obs <= threshold
                    high = obs > threshold

                    # ---------------- Low flow ----------------
                    if np.sum(low) > 2:

                        metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
                        metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))

                        metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
                        metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))

                        metrics_old["AbsBias_low"].append(
                            calc_abs_bias(obs[low], old[low])
                        )
                        metrics_new["AbsBias_low"].append(
                            calc_abs_bias(obs[low], new[low])
                        )

                    # ---------------- High flow ----------------
                    if np.sum(high) > 2:

                        metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
                        metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))

                        metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
                        metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))

                        metrics_old["AbsBias_high"].append(
                            calc_abs_bias(obs[high], old[high])
                        )
                        metrics_new["AbsBias_high"].append(
                            calc_abs_bias(obs[high], new[high])
                        )
                    # print("\n" + "=" * 60)
        # print(f"Temporal Test Total Flow COMPARISON for ({model}) using {method} for Baseflow Separation (1980-2014)")
        #                 # print("=" * 60)
                    
        # metric_order = [
        #                     "NSE",
        #                     "KGE",
        #                     "Bias",
        #                     "AbsBias",
        #                     "NSE_low",
        #                     "NSE_high",
        #                     "Bias_low",
        #                     "Bias_high",
        #                     "AbsBias_low",
        #                     "AbsBias_high",
        #                 ]
                    
        # for metric in metric_order:
        #                     old = np.nanmedian(metrics_old[metric])
        #                     new = np.nanmedian(metrics_new[metric])
        #                     print(f"{metric:<13}: {old:8.4f} → {new:8.4f}")                    
        # print("=" * 60)

(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)
(12418, 671)


In [2]:
method= 'LocalHYSEP'
methods= [ 'Furey','USGS', 'Eckhardt', 'LocalHYSEP']
models= ['HBV', 'HBV_capillary', 'marrmot_PRMS']
model= 'HBV'
import numpy as np

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den


def calc_kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    std_obs = np.std(obs)
    std_sim = np.std(sim)

    if std_obs == 0:
        return np.nan

    alpha = std_sim / std_obs

    mean_obs = np.mean(obs)
    if mean_obs == 0:
        return np.nan

    beta = np.mean(sim) / mean_obs

    return 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )


def calc_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return (np.sum(sim) - den) / den


def calc_abs_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return np.abs(np.sum(sim) - den) / den


for method in methods:
    for model in models:
        if model=='HBV':
                dyn= 'parBETA_parBETAET_'

        if model=='marrmot_PRMS':
                dyn= 'alpha_cgw_scx_tt_'

        if model=='HBV_capillary':
                dyn= 'parBETA_parBETAET_parK0_parUZL_'


        obs_flow= np.load(f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/baseflow.npy")
        single= np.load(f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/OUTPUTS_Temporal_Single/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/gwflow.npy")
        single=single.squeeze()
        multi= np.load(f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/gwflow.npy")
        multi= multi.squeeze()


        # ============================================================
        # Containers
        # ============================================================

        metrics_old = {
                "NSE": [],
                "KGE": [],
                "Bias": [],
                "AbsBias": [],
                "NSE_low": [],
                "NSE_high": [],
                "Bias_low": [],
                "Bias_high": [],
                "AbsBias_low": [],
                "AbsBias_high": [],
            }

        metrics_new = {
                "NSE": [],
                "KGE": [],
                "Bias": [],
                "AbsBias": [],
                "NSE_low": [],
                "NSE_high": [],
                "Bias_low": [],
                "Bias_high": [],
                "AbsBias_low": [],
                "AbsBias_high": [],
            }
        # --------------------------------------------------------
        # Loop over basins
        # --------------------------------------------------------
        for k in range(obs_flow.shape[1]):

                    obs = obs_flow[:, k]
                    old = single[:, k]
                    new = multi[:, k]

                    # =====================================================
                    # Overall metrics
                    # =====================================================
                    metrics_old["NSE"].append(calc_nse(obs, old))
                    metrics_new["NSE"].append(calc_nse(obs, new))

                    metrics_old["KGE"].append(calc_kge(obs, old))
                    metrics_new["KGE"].append(calc_kge(obs, new))

                    metrics_old["Bias"].append(calc_bias(obs, old))
                    metrics_new["Bias"].append(calc_bias(obs, new))

                    metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
                    metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

                    # =====================================================
                    # Low / High flow
                    # =====================================================
                    threshold = np.median(obs)

                    low = obs <= threshold
                    high = obs > threshold

                    # ---------------- Low flow ----------------
                    if np.sum(low) > 2:

                        metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
                        metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))

                        metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
                        metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))

                        metrics_old["AbsBias_low"].append(
                            calc_abs_bias(obs[low], old[low])
                        )
                        metrics_new["AbsBias_low"].append(
                            calc_abs_bias(obs[low], new[low])
                        )

                    # ---------------- High flow ----------------
                    if np.sum(high) > 2:

                        metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
                        metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))

                        metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
                        metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))

                        metrics_old["AbsBias_high"].append(
                            calc_abs_bias(obs[high], old[high])
                        )
                        metrics_new["AbsBias_high"].append(
                            calc_abs_bias(obs[high], new[high])
                        )
                    # print("\n" + "=" * 60)
        print(f"Temporal Test Baseflow COMPARISON for ({model}) using {method} for Baseflow Separation (1980-2014)")
                        # print("=" * 60)
                    
        metric_order = [
                            "NSE",
                            "KGE",
                            "Bias",
                            "AbsBias",
                            "NSE_low",
                            "NSE_high",
                            "Bias_low",
                            "Bias_high",
                            "AbsBias_low",
                            "AbsBias_high",
                        ]
                    
        for metric in metric_order:
                            old = np.nanmedian(metrics_old[metric])
                            new = np.nanmedian(metrics_new[metric])
                            print(f"{metric:<13}: {old:8.4f} → {new:8.4f}")
                    
        print("=" * 60)

Temporal Test Baseflow COMPARISON for (HBV) using Furey for Baseflow Separation (1980-2014)
NSE          :   0.4953 →   0.7684
KGE          :   0.5541 →   0.7926
Bias         :   0.2184 →   0.0390
AbsBias      :   0.2640 →   0.0862
NSE_low      :  -5.3271 →  -1.7601
NSE_high     :   0.2810 →   0.6313
Bias_low     :   0.7401 →   0.2951
Bias_high    :   0.0818 →  -0.0111
AbsBias_low  :   0.7401 →   0.3043
AbsBias_high :   0.1941 →   0.0752
Temporal Test Baseflow COMPARISON for (HBV_capillary) using Furey for Baseflow Separation (1980-2014)
NSE          :   0.4794 →   0.7721
KGE          :   0.4402 →   0.7842
Bias         :  -0.1325 →   0.0231
AbsBias      :   0.2714 →   0.0939
NSE_low      :  -1.6443 →  -1.7214
NSE_high     :   0.2230 →   0.6204
Bias_low     :   0.3043 →   0.2720
Bias_high    :  -0.2535 →  -0.0283
AbsBias_low  :   0.4232 →   0.3049
AbsBias_high :   0.3138 →   0.0816
Temporal Test Baseflow COMPARISON for (marrmot_PRMS) using Furey for Baseflow Separation (1980-2014)
NSE  

In [4]:
["Year", "Mnth", "Day", "Hr", "dayl(s)", "prcp(mm/day)", "srad(W/m2)", "swe(mm)", "tmax(C)", "tmin(C)", "vp(Pa)", "PEVAP", "00060_Mean", "tmean(C)", "gage_id", "baseflow"][14]

'gage_id'

In [ ]:
###Excluding Training Period
# ============================================================
# Time Range Configuration (Post-Warmup Array: 1981-01-01 to 2014-12-31)
# ============================================================
TEST_START_IDX = 0  # Start of 12,418 array (Jan 1, 1981)
TEST_END_IDX = 12418  # End of 12,418 array (Dec 31, 2014)

# Training period to exclude (shifted by -365 days due to warmup removal):
# 1989-10-01 to 1999-09-30
EXCLUDE_START_IDX = 3195
EXCLUDE_END_IDX = 6847  # exclusive upper bound for np.arange

methods = ["Furey", "USGS", "Eckhardt", "LocalHYSEP"]
models = ["HBV", "HBV_capillary", "marrmot_PRMS"]

import numpy as np
import pandas as pd
for method in methods:
  for model in models:
    if model == "HBV":
      dyn = "parBETA_parBETAET_"
    elif model == "marrmot_PRMS":
      dyn = "alpha_cgw_scx_tt_"
    elif model == "HBV_capillary":
      dyn = "parBETA_parBETAET_parK0_parUZL_"

    gauge= np.load("/scratch/data/CAMELS_FORCING/daymet_19800101_20141231_forcing_671_baseflow_mmday.npy")
    gid= gauge[:,0,14]
    g531=pd.read_csv("/scratch/data/CAMELS_531.csv")['Gauge ID']
    obs_flow = np.load(
        f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/00060_Mean.npy"
    )
    single = np.load(
        f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/OUTPUTS_Temporal_Single/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/flow_sim.npy"
    )
    single = single.squeeze()
    multi = np.load(
        f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/flow_sim.npy"
    )
    multi = multi.squeeze()

    # Optional: If your loaded numpy arrays are already sliced to length 12,418
    # (i.e. you already dropped the first 365 rows), make sure TEST_END matches obs_flow.shape[0].
    # Assuming obs_flow has shape (12418, basins):
    actual_len = obs_flow.shape[0]
    if actual_len == 12418:
      # Arrays already lack the warmup year
      base_indices = np.arange(0, 12418)
      # Adjust exclusion indices since 0 is now Jan 1, 1981
      exclude_indices = np.arange(3195, 6847)
    else:
      # Arrays are the full 12784 length (including 1980 warmup)
      base_indices = np.arange(365, 12784)
      exclude_indices = np.arange(3195 + 365, 6847 + 365)

    valid_indices = np.setdiff1d(base_indices, exclude_indices)

    obs_flow = obs_flow[valid_indices, :]
    single = single[valid_indices, :]
    multi = multi[valid_indices, :]

    # Containers
    metrics_old = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    metrics_new = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    # --------------------------------------------------------
    # Loop over basins
    # --------------------------------------------------------
    for k in range(obs_flow.shape[1]):
      obs = obs_flow[:, k]
      old = single[:, k]
      new = multi[:, k]

      # Overall metrics
      metrics_old["NSE"].append(calc_nse(obs, old))
      metrics_new["NSE"].append(calc_nse(obs, new))

      metrics_old["KGE"].append(calc_kge(obs, old))
      metrics_new["KGE"].append(calc_kge(obs, new))

      metrics_old["Bias"].append(calc_bias(obs, old))
      metrics_new["Bias"].append(calc_bias(obs, new))

      metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
      metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

      # Low / High flow
      threshold = np.median(obs)
      low = obs <= threshold
      high = obs > threshold

      if np.sum(low) > 2:
        metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
        metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))
        metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
        metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))
        metrics_old["AbsBias_low"].append(calc_abs_bias(obs[low], old[low]))
        metrics_new["AbsBias_low"].append(calc_abs_bias(obs[low], new[low]))

      if np.sum(high) > 2:
        metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
        metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))
        metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
        metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))
        metrics_old["AbsBias_high"].append(calc_abs_bias(obs[high], old[high]))
        metrics_new["AbsBias_high"].append(calc_abs_bias(obs[high], new[high]))

    print("\n" + "=" * 60)
    print(
        f"Temporal Test ({model}) Total Flow| Method: {method} | Out-of-Sample Eval"
        f" Days: {len(valid_indices)}"
    )
    print("=" * 60)

    metric_order = [
        "NSE",
        "KGE",
        "Bias",
        "AbsBias",
        "NSE_low",
        "NSE_high",
        "Bias_low",
        "Bias_high",
        "AbsBias_low",
        "AbsBias_high",
    ]

    for metric in metric_order:
      old_val = np.nanmedian(metrics_old[metric])
      new_val = np.nanmedian(metrics_new[metric])
      print(f"{metric:<13}: {old_val:8.4f} → {new_val:8.4f}")
    print("=" * 60)


Temporal Test (HBV) Total Flow| Method: Furey | Out-of-Sample Eval Days: 8766
NSE          :   0.5863 →   0.5788
KGE          :   0.6369 →   0.6242
Bias         :   0.0341 →   0.0151
AbsBias      :   0.1018 →   0.1019
NSE_low      :  -0.5022 →  -0.5131
NSE_high     :   0.6725 →   0.6680
Bias_low     :   0.0786 →   0.0546
Bias_high    :  -0.0311 →  -0.0444
AbsBias_low  :   0.1430 →   0.1077
AbsBias_high :   0.0947 →   0.1030

Temporal Test (HBV_capillary) Total Flow| Method: Furey | Out-of-Sample Eval Days: 8766
NSE          :   0.5884 →   0.5918
KGE          :   0.6333 →   0.6210
Bias         :   0.0197 →  -0.0036
AbsBias      :   0.0919 →   0.0917
NSE_low      :  -0.4685 →  -0.4671
NSE_high     :   0.6730 →   0.6741
Bias_low     :   0.0947 →   0.0573
Bias_high    :  -0.0437 →  -0.0616
AbsBias_low  :   0.1388 →   0.1011
AbsBias_high :   0.0930 →   0.1053

Temporal Test (marrmot_PRMS) Total Flow| Method: Furey | Out-of-Sample Eval Days: 8766
NSE          :   0.5710 →   0.5608
KGE      

In [6]:
###Excluding Training Period
# ============================================================
# Time Range Configuration (Post-Warmup Array: 1981-01-01 to 2014-12-31)
# ============================================================
TEST_START_IDX = 0  # Start of 12,418 array (Jan 1, 1981)
TEST_END_IDX = 12418  # End of 12,418 array (Dec 31, 2014)

# Training period to exclude (shifted by -365 days due to warmup removal):
# 1989-10-01 to 1999-09-30
EXCLUDE_START_IDX = 3195
EXCLUDE_END_IDX = 6847  # exclusive upper bound for np.arange

methods = ["Furey", "USGS", "Eckhardt", "LocalHYSEP"]
models = ["HBV", "HBV_capillary", "marrmot_PRMS"]

for method in methods:
  for model in models:
    if model == "HBV":
      dyn = "parBETA_parBETAET_"
    elif model == "marrmot_PRMS":
      dyn = "alpha_cgw_scx_tt_"
    elif model == "HBV_capillary":
      dyn = "parBETA_parBETAET_parK0_parUZL_"

    obs_flow = np.load(
        f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/baseflow.npy"
    )
    single = np.load(
        f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/OUTPUTS_Temporal_Single/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/gwflow.npy"
    )
    single = single.squeeze()
    multi = np.load(
        f"/home/s0shar47/scratch/OUTPUTS_Paper1/Temporal/{method}_Multi/LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1980_2014/gwflow.npy"
    )
    multi = multi.squeeze()

    # Optional: If your loaded numpy arrays are already sliced to length 12,418
    # (i.e. you already dropped the first 365 rows), make sure TEST_END matches obs_flow.shape[0].
    # Assuming obs_flow has shape (12418, basins):
    actual_len = obs_flow.shape[0]
    if actual_len == 12418:
      # Arrays already lack the warmup year
      base_indices = np.arange(0, 12418)
      # Adjust exclusion indices since 0 is now Jan 1, 1981
      exclude_indices = np.arange(3195, 6847)
    else:
      # Arrays are the full 12784 length (including 1980 warmup)
      base_indices = np.arange(365, 12784)
      exclude_indices = np.arange(3195 + 365, 6847 + 365)

    valid_indices = np.setdiff1d(base_indices, exclude_indices)

    obs_flow = obs_flow[valid_indices, :]
    single = single[valid_indices, :]
    multi = multi[valid_indices, :]

    # Containers
    metrics_old = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    metrics_new = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    # --------------------------------------------------------
    # Loop over basins
    # --------------------------------------------------------
    for k in range(obs_flow.shape[1]):
      obs = obs_flow[:, k]
      old = single[:, k]
      new = multi[:, k]

      # Overall metrics
      metrics_old["NSE"].append(calc_nse(obs, old))
      metrics_new["NSE"].append(calc_nse(obs, new))

      metrics_old["KGE"].append(calc_kge(obs, old))
      metrics_new["KGE"].append(calc_kge(obs, new))

      metrics_old["Bias"].append(calc_bias(obs, old))
      metrics_new["Bias"].append(calc_bias(obs, new))

      metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
      metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

      # Low / High flow
      threshold = np.median(obs)
      low = obs <= threshold
      high = obs > threshold

      if np.sum(low) > 2:
        metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
        metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))
        metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
        metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))
        metrics_old["AbsBias_low"].append(calc_abs_bias(obs[low], old[low]))
        metrics_new["AbsBias_low"].append(calc_abs_bias(obs[low], new[low]))

      if np.sum(high) > 2:
        metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
        metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))
        metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
        metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))
        metrics_old["AbsBias_high"].append(calc_abs_bias(obs[high], old[high]))
        metrics_new["AbsBias_high"].append(calc_abs_bias(obs[high], new[high]))

    print("\n" + "=" * 60)
    print(
        f"Temporal Test ({model}) Baseflow| Method: {method} | Out-of-Sample Eval"
        f" Days: {len(valid_indices)}"
    )
    print("=" * 60)

    metric_order = [
        "NSE",
        "KGE",
        "Bias",
        "AbsBias",
        "NSE_low",
        "NSE_high",
        "Bias_low",
        "Bias_high",
        "AbsBias_low",
        "AbsBias_high",
    ]

    for metric in metric_order:
      old_val = np.nanmedian(metrics_old[metric])
      new_val = np.nanmedian(metrics_new[metric])
      print(f"{metric:<13}: {old_val:8.4f} → {new_val:8.4f}")
    print("=" * 60)


Temporal Test (HBV) Baseflow| Method: Furey | Out-of-Sample Eval Days: 8766
NSE          :   0.4697 →   0.7445
KGE          :   0.5344 →   0.7705
Bias         :   0.2348 →   0.0491
AbsBias      :   0.2743 →   0.1045
NSE_low      :  -6.0426 →  -2.0928
NSE_high     :   0.2495 →   0.6090
Bias_low     :   0.7607 →   0.3088
Bias_high    :   0.0898 →  -0.0112
AbsBias_low  :   0.7607 →   0.3248
AbsBias_high :   0.2135 →   0.0821

Temporal Test (HBV_capillary) Baseflow| Method: Furey | Out-of-Sample Eval Days: 8766
NSE          :   0.4723 →   0.7488
KGE          :   0.4306 →   0.7652
Bias         :  -0.1146 →   0.0363
AbsBias      :   0.2825 →   0.1064
NSE_low      :  -1.6731 →  -1.7663
NSE_high     :   0.1948 →   0.5998
Bias_low     :   0.3427 →   0.2917
Bias_high    :  -0.2341 →  -0.0206
AbsBias_low  :   0.4404 →   0.3159
AbsBias_high :   0.3201 →   0.0895

Temporal Test (marrmot_PRMS) Baseflow| Method: Furey | Out-of-Sample Eval Days: 8766
NSE          :   0.3614 →   0.7186
KGE          : 

In [9]:
# ============================================================
# Excluding Training Period
# Evaluate ONLY the 531 CAMELS gauges
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def calc_nse(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum(
        (obs - np.mean(obs)) ** 2
    )

    if den == 0:
        return np.nan

    return (
        1
        - np.sum((obs - sim) ** 2) / den
    )


def calc_kge(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    obs_mean = np.mean(obs)
    sim_mean = np.mean(sim)

    obs_std = np.std(obs)
    sim_std = np.std(sim)

    if obs_std == 0 or obs_mean == 0:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    if not np.isfinite(r):
        return np.nan

    alpha = sim_std / obs_std
    beta = sim_mean / obs_mean

    return (
        1
        - np.sqrt(
            (r - 1) ** 2
            + (alpha - 1) ** 2
            + (beta - 1) ** 2
        )
    )


def calc_bias(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) == 0:
        return np.nan

    return np.mean(sim - obs)


def calc_abs_bias(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) == 0:
        return np.nan

    return np.mean(np.abs(sim - obs))


# ============================================================
# SETTINGS
# ============================================================

methods = [
    "Furey",
    "USGS",
    "Eckhardt",
    "LocalHYSEP"
]

models = [
    "HBV",
    "HBV_capillary",
    "marrmot_PRMS"
]


# ============================================================
# TIME RANGE CONFIGURATION
#
# Post-warmup array:
# 1981-01-01 to 2014-12-31
#
# Training period:
# 1989-10-01 to 1999-09-30
#
# Shifted by -365 days:
# 1989-10-01 -> index 3195
# 1999-09-30 -> index 6846
#
# End index is exclusive.
# ============================================================

TEST_START_IDX = 0
TEST_END_IDX = 12418

EXCLUDE_START_IDX = 3195
EXCLUDE_END_IDX = 6847


# ============================================================
# LOAD GAUGE IDS
# ============================================================

forcing = np.load(
    "/scratch/data/CAMELS_FORCING/"
    "daymet_19800101_20141231_forcing_671_baseflow_mmday.npy"
)

# Gauge ID is variable 14
gid = forcing[:, 0, 14]

# Convert to integer for safe comparison
gid = gid.astype(int)


# ============================================================
# LOAD THE 531 GAUGE IDS
# ============================================================

g531 = pd.read_csv(
    "/scratch/data/CAMELS_531.csv"
)["Gauge ID"]

g531 = g531.astype(int).values


# ============================================================
# FIND THE 531 BASINS
# ============================================================

basin_mask = np.isin(
    gid,
    g531
)


# # ============================================================
# # CHECK GAUGE MATCHING
# # ============================================================

# print("\n" + "=" * 70)
# print("GAUGE SELECTION")
# print("=" * 70)

# print(
#     f"Total gauges in forcing file : {len(gid)}"
# )

# print(
#     f"Number of gauges in CSV      : {len(g531)}"
# )

# print(
#     f"Matched gauges               : {np.sum(basin_mask)}"
# )

# missing_gauges = g531[
#     ~np.isin(g531, gid)
# ]

# # print(
# #     f"Missing gauges               : {len(missing_gauges)}"
# # )

# if len(missing_gauges) > 0:

#     print(
#         "Missing Gauge IDs:"
#     )

#     print(
#         missing_gauges
#     )

# print("=" * 70)


# ============================================================
# CHECK THAT ALL 531 GAUGES WERE FOUND
# ============================================================

if np.sum(basin_mask) != len(g531):

    raise ValueError(
        f"Expected {len(g531)} gauges, "
        f"but only {np.sum(basin_mask)} were found."
    )


# ============================================================
# LOOP OVER METHODS AND MODELS
# ============================================================

for method in methods:

    for model in models:

        # ====================================================
        # MODEL DYNAMIC PARAMETERS
        # ====================================================

        if model == "HBV":

            dyn = (
                "parBETA_parBETAET_"
            )

        elif model == "marrmot_PRMS":

            dyn = (
                "alpha_cgw_scx_tt_"
            )

        elif model == "HBV_capillary":

            dyn = (
                "parBETA_parBETAET_"
                "parK0_parUZL_"
            )

        else:

            raise ValueError(
                f"Unknown model: {model}"
            )


        # ====================================================
        # FILE PATHS
        # ====================================================

        obs_path = (
            f"/home/s0shar47/scratch/"
            f"OUTPUTS_Paper1/Temporal/"
            f"{method}_Multi/"
            f"LSTM_{model}_E50_R365_B100_H256_"
            f"tr1989_1999_n16_111111/"
            f"{dyn}/ts1980_2014/"
            f"00060_Mean.npy"
        )


        single_path = (
            "/home/s0shar47/scratch/"
            "OUTPUTS_Paper1/Temporal/"
            "OUTPUTS_Temporal_Single/"
            f"LSTM_{model}_E50_R365_B100_H256_"
            f"tr1989_1999_n16_111111/"
            f"{dyn}/ts1980_2014/"
            f"flow_sim.npy"
        )


        multi_path = (
            f"/home/s0shar47/scratch/"
            f"OUTPUTS_Paper1/Temporal/"
            f"{method}_Multi/"
            f"LSTM_{model}_E50_R365_B100_H256_"
            f"tr1989_1999_n16_111111/"
            f"{dyn}/ts1980_2014/"
            f"flow_sim.npy"
        )


        # ====================================================
        # LOAD DATA
        # ====================================================

        obs_flow = np.load(
            obs_path
        )

        single = np.load(
            single_path
        ).squeeze()

        multi = np.load(
            multi_path
        ).squeeze()


        # ====================================================
        # MAKE SURE ARRAYS ARE 2-D
        # ====================================================

        if obs_flow.ndim == 1:
            obs_flow = obs_flow[:, None]

        if single.ndim == 1:
            single = single[:, None]

        if multi.ndim == 1:
            multi = multi[:, None]


        # ====================================================
        # PRINT ORIGINAL SHAPES
        # ====================================================

        print("\n")
        print("=" * 70)
        print(
            f"Method: {method} | Model: {model}"
        )
        print("=" * 70)

        # print(
        #     f"Original obs shape    : {obs_flow.shape}"
        # )

        # print(
        #     f"Original single shape : {single.shape}"
        # )

        # print(
        #     f"Original multi shape  : {multi.shape}"
        # )


        # ====================================================
        # CHECK BASIN DIMENSION
        # ====================================================

        if obs_flow.shape[1] != len(gid):

            raise ValueError(
                "\nThe basin dimension of obs_flow does not "
                "match the 671-gauge forcing file.\n"
                f"obs_flow basins = {obs_flow.shape[1]}\n"
                f"forcing gauges = {len(gid)}\n\n"
                "The code assumes that the columns of obs_flow, "
                "single, and multi correspond to the 671 gauges "
                "in the forcing file."
            )


        if single.shape[1] != len(gid):

            raise ValueError(
                f"Single has {single.shape[1]} basins, "
                f"but forcing has {len(gid)} gauges."
            )


        if multi.shape[1] != len(gid):

            raise ValueError(
                f"Multi has {multi.shape[1]} basins, "
                f"but forcing has {len(gid)} gauges."
            )


        # ====================================================
        # SELECT ONLY THE 531 GAUGES
        # ====================================================

        obs_flow = obs_flow[
            :,
            basin_mask
        ]

        single = single[
            :,
            basin_mask
        ]

        multi = multi[
            :,
            basin_mask
        ]


        # ====================================================
        # CHECK SELECTED SHAPES
        # ====================================================

        # print(
        #     f"531-gauge obs shape    : {obs_flow.shape}"
        # )

        # print(
        #     f"531-gauge single shape : {single.shape}"
        # )

        # print(
        #     f"531-gauge multi shape  : {multi.shape}"
        # )


        # ====================================================
        # TIME RANGE
        # ====================================================

        actual_len = obs_flow.shape[0]


        # ----------------------------------------------------
        # If arrays already start at 1981
        # ----------------------------------------------------

        if actual_len == 12418:

            base_indices = np.arange(
                TEST_START_IDX,
                TEST_END_IDX
            )

            exclude_indices = np.arange(
                EXCLUDE_START_IDX,
                EXCLUDE_END_IDX
            )


        # ----------------------------------------------------
        # If arrays contain 1980 warmup
        # ----------------------------------------------------

        else:

            base_indices = np.arange(
                365,
                actual_len
            )

            exclude_indices = np.arange(
                EXCLUDE_START_IDX + 365,
                EXCLUDE_END_IDX + 365
            )


        # ====================================================
        # MAKE SURE INDICES ARE VALID
        # ====================================================

        max_len = min(
            obs_flow.shape[0],
            single.shape[0],
            multi.shape[0]
        )


        base_indices = base_indices[
            base_indices < max_len
        ]

        exclude_indices = exclude_indices[
            exclude_indices < max_len
        ]


        # ====================================================
        # REMOVE TRAINING PERIOD
        # ====================================================

        valid_indices = np.setdiff1d(
            base_indices,
            exclude_indices
        )


        # ====================================================
        # SLICE TIME
        # ====================================================

        obs_flow = obs_flow[
            valid_indices,
            :
        ]

        single = single[
            valid_indices,
            :
        ]

        multi = multi[
            valid_indices,
            :
        ]


        # ====================================================
        # METRIC CONTAINERS
        # ====================================================

        metrics_old = {

            "NSE": [],
            "KGE": [],
            "Bias": [],
            "AbsBias": [],

            "NSE_low": [],
            "NSE_high": [],

            "Bias_low": [],
            "Bias_high": [],

            "AbsBias_low": [],
            "AbsBias_high": []
        }


        metrics_new = {

            "NSE": [],
            "KGE": [],
            "Bias": [],
            "AbsBias": [],

            "NSE_low": [],
            "NSE_high": [],

            "Bias_low": [],
            "Bias_high": [],

            "AbsBias_low": [],
            "AbsBias_high": []
        }


        # ====================================================
        # LOOP OVER 531 BASINS ONLY
        # ====================================================

        for k in range(
            obs_flow.shape[1]
        ):

            obs = obs_flow[:, k]

            old = single[:, k]

            new = multi[:, k]


            # =================================================
            # OVERALL METRICS
            # =================================================

            metrics_old["NSE"].append(
                calc_nse(
                    obs,
                    old
                )
            )

            metrics_new["NSE"].append(
                calc_nse(
                    obs,
                    new
                )
            )


            metrics_old["KGE"].append(
                calc_kge(
                    obs,
                    old
                )
            )

            metrics_new["KGE"].append(
                calc_kge(
                    obs,
                    new
                )
            )


            metrics_old["Bias"].append(
                calc_bias(
                    obs,
                    old
                )
            )

            metrics_new["Bias"].append(
                calc_bias(
                    obs,
                    new
                )
            )


            metrics_old["AbsBias"].append(
                calc_abs_bias(
                    obs,
                    old
                )
            )

            metrics_new["AbsBias"].append(
                calc_abs_bias(
                    obs,
                    new
                )
            )


            # =================================================
            # LOW / HIGH FLOW
            # =================================================

            threshold = np.median(
                obs
            )

            low = (
                obs <= threshold
            )

            high = (
                obs > threshold
            )


            # =================================================
            # LOW FLOW
            # =================================================

            if np.sum(low) > 2:

                metrics_old["NSE_low"].append(
                    calc_nse(
                        obs[low],
                        old[low]
                    )
                )

                metrics_new["NSE_low"].append(
                    calc_nse(
                        obs[low],
                        new[low]
                    )
                )


                metrics_old["Bias_low"].append(
                    calc_bias(
                        obs[low],
                        old[low]
                    )
                )

                metrics_new["Bias_low"].append(
                    calc_bias(
                        obs[low],
                        new[low]
                    )
                )


                metrics_old["AbsBias_low"].append(
                    calc_abs_bias(
                        obs[low],
                        old[low]
                    )
                )

                metrics_new["AbsBias_low"].append(
                    calc_abs_bias(
                        obs[low],
                        new[low]
                    )
                )


            # =================================================
            # HIGH FLOW
            # =================================================

            if np.sum(high) > 2:

                metrics_old["NSE_high"].append(
                    calc_nse(
                        obs[high],
                        old[high]
                    )
                )

                metrics_new["NSE_high"].append(
                    calc_nse(
                        obs[high],
                        new[high]
                    )
                )


                metrics_old["Bias_high"].append(
                    calc_bias(
                        obs[high],
                        old[high]
                    )
                )

                metrics_new["Bias_high"].append(
                    calc_bias(
                        obs[high],
                        new[high]
                    )
                )


                metrics_old["AbsBias_high"].append(
                    calc_abs_bias(
                        obs[high],
                        old[high]
                    )
                )

                metrics_new["AbsBias_high"].append(
                    calc_abs_bias(
                        obs[high],
                        new[high]
                    )
                )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print("\n" + "=" * 60)

        print(
            f"Temporal Test ({model}) Total Flow | "
            f"Method: {method} | "
            f"531 Gauges | "
            f"Out-of-Sample Days: {len(valid_indices)}"
        )

        print("=" * 60)


        metric_order = [

            "NSE",
            "KGE",
            "Bias",
            "AbsBias",

            "NSE_low",
            "NSE_high",

            "Bias_low",
            "Bias_high",

            "AbsBias_low",
            "AbsBias_high"
        ]


        for metric in metric_order:

            old_val = np.nanmedian(
                metrics_old[metric]
            )

            new_val = np.nanmedian(
                metrics_new[metric]
            )

            print(
                f"{metric:<13}: "
                f"{old_val:8.4f} → "
                f"{new_val:8.4f}"
            )


        print("=" * 60)



Method: Furey | Model: HBV

Temporal Test (HBV) Total Flow | Method: Furey | 531 Gauges | Out-of-Sample Days: 8766
NSE          :   0.6131 →   0.6116
KGE          :   0.6595 →   0.6515
Bias         :   0.0902 →   0.0729
AbsBias      :   0.5383 →   0.5403
NSE_low      :  -0.4867 →  -0.4993
NSE_high     :   0.6804 →   0.6771
Bias_low     :   0.1884 →   0.1743
Bias_high    :  -0.0340 →  -0.0483
AbsBias_low  :   0.2894 →   0.2557
AbsBias_high :   0.7044 →   0.7072


Method: Furey | Model: HBV_capillary

Temporal Test (HBV_capillary) Total Flow | Method: Furey | 531 Gauges | Out-of-Sample Days: 8766
NSE          :   0.6137 →   0.6156
KGE          :   0.6649 →   0.6545
Bias         :   0.0788 →   0.0520
AbsBias      :   0.5169 →   0.5185
NSE_low      :  -0.4450 →  -0.4543
NSE_high     :   0.6853 →   0.6829
Bias_low     :   0.1916 →   0.1790
Bias_high    :  -0.0612 →  -0.0755
AbsBias_low  :   0.2677 →   0.2541
AbsBias_high :   0.6757 →   0.6847


Method: Furey | Model: marrmot_PRMS

Tempora

In [1]:
# ============================================================
# Excluding Training Period
# Evaluate ONLY the 531 CAMELS gauges
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def calc_nse(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum(
        (obs - np.mean(obs)) ** 2
    )

    if den == 0:
        return np.nan

    return (
        1
        - np.sum((obs - sim) ** 2) / den
    )


def calc_kge(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    obs_mean = np.mean(obs)
    sim_mean = np.mean(sim)

    obs_std = np.std(obs)
    sim_std = np.std(sim)

    if obs_std == 0 or obs_mean == 0:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    if not np.isfinite(r):
        return np.nan

    alpha = sim_std / obs_std
    beta = sim_mean / obs_mean

    return (
        1
        - np.sqrt(
            (r - 1) ** 2
            + (alpha - 1) ** 2
            + (beta - 1) ** 2
        )
    )


def calc_bias(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) == 0:
        return np.nan

    return np.mean(sim - obs)


def calc_abs_bias(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) == 0:
        return np.nan

    return np.mean(np.abs(sim - obs))


# ============================================================
# SETTINGS
# ============================================================

methods = [
    "Furey",
    "USGS",
    "Eckhardt",
    "LocalHYSEP"
]

models = [
    "HBV",
    "HBV_capillary",
    "marrmot_PRMS"
]


# ============================================================
# TIME RANGE CONFIGURATION
#
# Post-warmup array:
# 1981-01-01 to 2014-12-31
#
# Training period:
# 1989-10-01 to 1999-09-30
#
# Shifted by -365 days:
# 1989-10-01 -> index 3195
# 1999-09-30 -> index 6846
#
# End index is exclusive.
# ============================================================

TEST_START_IDX = 0
TEST_END_IDX = 12418

EXCLUDE_START_IDX = 3195
EXCLUDE_END_IDX = 6847


# ============================================================
# LOAD GAUGE IDS
# ============================================================

forcing = np.load(
    "/scratch/data/CAMELS_FORCING/"
    "daymet_19800101_20141231_forcing_671_baseflow_mmday.npy"
)

# Gauge ID is variable 14
gid = forcing[:, 0, 14]

# Convert to integer for safe comparison
gid = gid.astype(int)


# ============================================================
# LOAD THE 531 GAUGE IDS
# ============================================================

g531 = pd.read_csv(
    "/scratch/data/CAMELS_531.csv"
)["Gauge ID"]

g531 = g531.astype(int).values


# ============================================================
# FIND THE 531 BASINS
# ============================================================

basin_mask = np.isin(
    gid,
    g531
)


# # ============================================================
# # CHECK GAUGE MATCHING
# # ============================================================

# print("\n" + "=" * 70)
# print("GAUGE SELECTION")
# print("=" * 70)

# print(
#     f"Total gauges in forcing file : {len(gid)}"
# )

# print(
#     f"Number of gauges in CSV      : {len(g531)}"
# )

# print(
#     f"Matched gauges               : {np.sum(basin_mask)}"
# )

# missing_gauges = g531[
#     ~np.isin(g531, gid)
# ]

# # print(
# #     f"Missing gauges               : {len(missing_gauges)}"
# # )

# if len(missing_gauges) > 0:

#     print(
#         "Missing Gauge IDs:"
#     )

#     print(
#         missing_gauges
#     )

# print("=" * 70)


# ============================================================
# CHECK THAT ALL 531 GAUGES WERE FOUND
# ============================================================

if np.sum(basin_mask) != len(g531):

    raise ValueError(
        f"Expected {len(g531)} gauges, "
        f"but only {np.sum(basin_mask)} were found."
    )


# ============================================================
# LOOP OVER METHODS AND MODELS
# ============================================================

for method in methods:

    for model in models:

        # ====================================================
        # MODEL DYNAMIC PARAMETERS
        # ====================================================

        if model == "HBV":

            dyn = (
                "parBETA_parBETAET_"
            )

        elif model == "marrmot_PRMS":

            dyn = (
                "alpha_cgw_scx_tt_"
            )

        elif model == "HBV_capillary":

            dyn = (
                "parBETA_parBETAET_"
                "parK0_parUZL_"
            )

        else:

            raise ValueError(
                f"Unknown model: {model}"
            )


        # ====================================================
        # FILE PATHS
        # ====================================================

        obs_path = (
            f"/home/s0shar47/scratch/"
            f"OUTPUTS_Paper1/Temporal/"
            f"{method}_Multi/"
            f"LSTM_{model}_E50_R365_B100_H256_"
            f"tr1989_1999_n16_111111/"
            f"{dyn}/ts1980_2014/"
            f"baseflow.npy"
        )


        single_path = (
            "/home/s0shar47/scratch/"
            "OUTPUTS_Paper1/Temporal/"
            "OUTPUTS_Temporal_Single/"
            f"LSTM_{model}_E50_R365_B100_H256_"
            f"tr1989_1999_n16_111111/"
            f"{dyn}/ts1980_2014/"
            f"gwflow.npy"
        )


        multi_path = (
            f"/home/s0shar47/scratch/"
            f"OUTPUTS_Paper1/Temporal/"
            f"{method}_Multi/"
            f"LSTM_{model}_E50_R365_B100_H256_"
            f"tr1989_1999_n16_111111/"
            f"{dyn}/ts1980_2014/"
            f"gwflow.npy"
        )


        # ====================================================
        # LOAD DATA
        # ====================================================

        obs_flow = np.load(
            obs_path
        )

        single = np.load(
            single_path
        ).squeeze()

        multi = np.load(
            multi_path
        ).squeeze()


        # ====================================================
        # MAKE SURE ARRAYS ARE 2-D
        # ====================================================

        if obs_flow.ndim == 1:
            obs_flow = obs_flow[:, None]

        if single.ndim == 1:
            single = single[:, None]

        if multi.ndim == 1:
            multi = multi[:, None]


        # ====================================================
        # PRINT ORIGINAL SHAPES
        # ====================================================

        print("\n")
        print("=" * 70)
        print(
            f"Method: {method} | Model: {model}"
        )
        print("=" * 70)

        # print(
        #     f"Original obs shape    : {obs_flow.shape}"
        # )

        # print(
        #     f"Original single shape : {single.shape}"
        # )

        # print(
        #     f"Original multi shape  : {multi.shape}"
        # )


        # ====================================================
        # CHECK BASIN DIMENSION
        # ====================================================

        if obs_flow.shape[1] != len(gid):

            raise ValueError(
                "\nThe basin dimension of obs_flow does not "
                "match the 671-gauge forcing file.\n"
                f"obs_flow basins = {obs_flow.shape[1]}\n"
                f"forcing gauges = {len(gid)}\n\n"
                "The code assumes that the columns of obs_flow, "
                "single, and multi correspond to the 671 gauges "
                "in the forcing file."
            )


        if single.shape[1] != len(gid):

            raise ValueError(
                f"Single has {single.shape[1]} basins, "
                f"but forcing has {len(gid)} gauges."
            )


        if multi.shape[1] != len(gid):

            raise ValueError(
                f"Multi has {multi.shape[1]} basins, "
                f"but forcing has {len(gid)} gauges."
            )


        # ====================================================
        # SELECT ONLY THE 531 GAUGES
        # ====================================================

        obs_flow = obs_flow[
            :,
            basin_mask
        ]

        single = single[
            :,
            basin_mask
        ]

        multi = multi[
            :,
            basin_mask
        ]


        # ====================================================
        # CHECK SELECTED SHAPES
        # ====================================================

        # print(
        #     f"531-gauge obs shape    : {obs_flow.shape}"
        # )

        # print(
        #     f"531-gauge single shape : {single.shape}"
        # )

        # print(
        #     f"531-gauge multi shape  : {multi.shape}"
        # )


        # ====================================================
        # TIME RANGE
        # ====================================================

        actual_len = obs_flow.shape[0]


        # ----------------------------------------------------
        # If arrays already start at 1981
        # ----------------------------------------------------

        if actual_len == 12418:

            base_indices = np.arange(
                TEST_START_IDX,
                TEST_END_IDX
            )

            exclude_indices = np.arange(
                EXCLUDE_START_IDX,
                EXCLUDE_END_IDX
            )


        # ----------------------------------------------------
        # If arrays contain 1980 warmup
        # ----------------------------------------------------

        else:

            base_indices = np.arange(
                365,
                actual_len
            )

            exclude_indices = np.arange(
                EXCLUDE_START_IDX + 365,
                EXCLUDE_END_IDX + 365
            )


        # ====================================================
        # MAKE SURE INDICES ARE VALID
        # ====================================================

        max_len = min(
            obs_flow.shape[0],
            single.shape[0],
            multi.shape[0]
        )


        base_indices = base_indices[
            base_indices < max_len
        ]

        exclude_indices = exclude_indices[
            exclude_indices < max_len
        ]


        # ====================================================
        # REMOVE TRAINING PERIOD
        # ====================================================

        valid_indices = np.setdiff1d(
            base_indices,
            exclude_indices
        )


        # ====================================================
        # SLICE TIME
        # ====================================================

        obs_flow = obs_flow[
            valid_indices,
            :
        ]

        single = single[
            valid_indices,
            :
        ]

        multi = multi[
            valid_indices,
            :
        ]


        # ====================================================
        # METRIC CONTAINERS
        # ====================================================

        metrics_old = {

            "NSE": [],
            "KGE": [],
            "Bias": [],
            "AbsBias": [],

            "NSE_low": [],
            "NSE_high": [],

            "Bias_low": [],
            "Bias_high": [],

            "AbsBias_low": [],
            "AbsBias_high": []
        }


        metrics_new = {

            "NSE": [],
            "KGE": [],
            "Bias": [],
            "AbsBias": [],

            "NSE_low": [],
            "NSE_high": [],

            "Bias_low": [],
            "Bias_high": [],

            "AbsBias_low": [],
            "AbsBias_high": []
        }


        # ====================================================
        # LOOP OVER 531 BASINS ONLY
        # ====================================================

        for k in range(
            obs_flow.shape[1]
        ):

            obs = obs_flow[:, k]

            old = single[:, k]

            new = multi[:, k]


            # =================================================
            # OVERALL METRICS
            # =================================================

            metrics_old["NSE"].append(
                calc_nse(
                    obs,
                    old
                )
            )

            metrics_new["NSE"].append(
                calc_nse(
                    obs,
                    new
                )
            )


            metrics_old["KGE"].append(
                calc_kge(
                    obs,
                    old
                )
            )

            metrics_new["KGE"].append(
                calc_kge(
                    obs,
                    new
                )
            )


            metrics_old["Bias"].append(
                calc_bias(
                    obs,
                    old
                )
            )

            metrics_new["Bias"].append(
                calc_bias(
                    obs,
                    new
                )
            )


            metrics_old["AbsBias"].append(
                calc_abs_bias(
                    obs,
                    old
                )
            )

            metrics_new["AbsBias"].append(
                calc_abs_bias(
                    obs,
                    new
                )
            )


            # =================================================
            # LOW / HIGH FLOW
            # =================================================

            threshold = np.median(
                obs
            )

            low = (
                obs <= threshold
            )

            high = (
                obs > threshold
            )


            # =================================================
            # LOW FLOW
            # =================================================

            if np.sum(low) > 2:

                metrics_old["NSE_low"].append(
                    calc_nse(
                        obs[low],
                        old[low]
                    )
                )

                metrics_new["NSE_low"].append(
                    calc_nse(
                        obs[low],
                        new[low]
                    )
                )


                metrics_old["Bias_low"].append(
                    calc_bias(
                        obs[low],
                        old[low]
                    )
                )

                metrics_new["Bias_low"].append(
                    calc_bias(
                        obs[low],
                        new[low]
                    )
                )


                metrics_old["AbsBias_low"].append(
                    calc_abs_bias(
                        obs[low],
                        old[low]
                    )
                )

                metrics_new["AbsBias_low"].append(
                    calc_abs_bias(
                        obs[low],
                        new[low]
                    )
                )


            # =================================================
            # HIGH FLOW
            # =================================================

            if np.sum(high) > 2:

                metrics_old["NSE_high"].append(
                    calc_nse(
                        obs[high],
                        old[high]
                    )
                )

                metrics_new["NSE_high"].append(
                    calc_nse(
                        obs[high],
                        new[high]
                    )
                )


                metrics_old["Bias_high"].append(
                    calc_bias(
                        obs[high],
                        old[high]
                    )
                )

                metrics_new["Bias_high"].append(
                    calc_bias(
                        obs[high],
                        new[high]
                    )
                )


                metrics_old["AbsBias_high"].append(
                    calc_abs_bias(
                        obs[high],
                        old[high]
                    )
                )

                metrics_new["AbsBias_high"].append(
                    calc_abs_bias(
                        obs[high],
                        new[high]
                    )
                )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print("\n" + "=" * 60)

        print(
            f"Temporal Test ({model}) Baseflow | "
            f"Method: {method} | "
            f"531 Gauges | "
            f"Out-of-Sample Days: {len(valid_indices)}"
        )

        print("=" * 60)


        metric_order = [

            "NSE",
            "KGE",
            "Bias",
            "AbsBias",

            "NSE_low",
            "NSE_high",

            "Bias_low",
            "Bias_high",

            "AbsBias_low",
            "AbsBias_high"
        ]


        for metric in metric_order:

            old_val = np.nanmedian(
                metrics_old[metric]
            )

            new_val = np.nanmedian(
                metrics_new[metric]
            )

            print(
                f"{metric:<13}: "
                f"{old_val:8.4f} → "
                f"{new_val:8.4f}"
            )


        print("=" * 60)



Method: Furey | Model: HBV

Temporal Test (HBV) Baseflow | Method: Furey | 531 Gauges | Out-of-Sample Days: 8766
NSE          :   0.4617 →   0.7523
KGE          :   0.5252 →   0.7778
Bias         :   0.0987 →   0.0171
AbsBias      :   0.2607 →   0.1529
NSE_low      :  -5.6565 →  -1.8491
NSE_high     :   0.2160 →   0.5981
Bias_low     :   0.0586 →   0.0254
Bias_high    :   0.0403 →  -0.0064
AbsBias_low  :   0.1084 →   0.0677
AbsBias_high :   0.3184 →   0.2032


Method: Furey | Model: HBV_capillary

Temporal Test (HBV_capillary) Baseflow | Method: Furey | 531 Gauges | Out-of-Sample Days: 8766
NSE          :   0.4625 →   0.7580
KGE          :   0.4112 →   0.7739
Bias         :  -0.0507 →   0.0130
AbsBias      :   0.2120 →   0.1508
NSE_low      :  -1.2536 →  -1.5621
NSE_high     :   0.1494 →   0.5925
Bias_low     :   0.0128 →   0.0216
Bias_high    :  -0.1698 →  -0.0265
AbsBias_low  :   0.0743 →   0.0687
AbsBias_high :   0.2742 →   0.2041


Method: Furey | Model: marrmot_PRMS

Temporal Te

In [11]:
obs= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight7525/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/00060_Mean.npy")
sim= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight7525/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/flow_sim.npy")
sim= sim.squeeze()
calc_nse(obs,sim)

np.float32(0.77339953)

In [12]:
obs= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight7525/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/baseflow.npy")
sim= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight7525/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/gwflow.npy")
sim= sim.squeeze()
calc_nse(obs,sim)

np.float32(0.7829015)

In [10]:
obs= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/00060_Mean.npy")
sim= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/flow_sim.npy")
sim= sim.squeeze()
calc_nse(obs,sim)

np.float32(0.76547)

In [13]:
obs= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/baseflow.npy")
sim= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/gwflow.npy")
sim= sim.squeeze()
calc_nse(obs,sim)

np.float32(0.7738576)

In [14]:
obs= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight2575/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/baseflow.npy")
sim= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight2575/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/gwflow.npy")
sim= sim.squeeze()
calc_nse(obs,sim)

np.float32(0.78277045)

In [15]:
obs= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight2575/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/00060_Mean.npy")
sim= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/Furey_Multi/1_modifiedlossweight2575/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/flow_sim.npy")
sim= sim.squeeze()
calc_nse(obs,sim)

np.float32(0.76633507)